In [ ]:
# import libraries
import os
import json
import pandas as pd
import numpy as np
import re
import torch
from transformers import AutoConfig
import matplotlib.pyplot as plt
import seaborn as sns

run_to_load = "microsoft_Phi-3-medium-128k-instruct_20250723_231551"


In [2]:
prompt_0_layer_0 = torch.load(f'/home/jcuello/emotion_drift/data/03_activations/activations_{run_to_load}/prompt_0_layer_0.pt')

print(prompt_0_layer_0.shape)

torch.Size([12, 5120])


In [ ]:
len(prompt_0_layer_0[-1]) # Last token

5120

In [ ]:
for i, file in enumerate(os.listdir(f'/home/jcuello/emotion_drift/data/03_activations/activations_{run_to_load}')):
    if file.endswith('.pt'):
        print(file)
        activations = torch.load(f'/home/jcuello/emotion_drift/data/03_activations/activations_{run_to_load}/{file}')
        print(activations.shape, i) # i should be = n_prompts*n_layers

prompt_6_layer_30.pt
torch.Size([13, 5120]) 0
prompt_2_layer_14.pt
torch.Size([14, 5120]) 1
prompt_9_layer_7.pt
torch.Size([13, 5120]) 2
prompt_9_layer_21.pt
torch.Size([13, 5120]) 3
prompt_0_layer_30.pt
torch.Size([12, 5120]) 4
prompt_1_layer_4.pt
torch.Size([11, 5120]) 5
prompt_9_layer_11.pt
torch.Size([13, 5120]) 6
prompt_7_layer_28.pt
torch.Size([10, 5120]) 7
prompt_0_layer_36.pt
torch.Size([12, 5120]) 8
prompt_0_layer_31.pt
torch.Size([12, 5120]) 9
prompt_9_layer_8.pt
torch.Size([13, 5120]) 10
prompt_1_layer_11.pt
torch.Size([11, 5120]) 11
prompt_3_layer_23.pt
torch.Size([15, 5120]) 12
prompt_3_layer_8.pt
torch.Size([15, 5120]) 13
prompt_8_layer_27.pt
torch.Size([9, 5120]) 14
prompt_9_layer_27.pt
torch.Size([13, 5120]) 15
prompt_9_layer_2.pt
torch.Size([13, 5120]) 16
prompt_9_layer_30.pt
torch.Size([13, 5120]) 17
prompt_9_layer_16.pt
torch.Size([13, 5120]) 18
prompt_9_layer_32.pt
torch.Size([13, 5120]) 19
prompt_2_layer_7.pt
torch.Size([14, 5120]) 20
prompt_0_layer_20.pt
torch.Siz

In [ ]:
# Should group the activations by prompt, one row per prompt, and one column per layer


In [7]:
json_file_path = f'/home/jcuello/emotion_drift/data/02_generated/outputs_{run_to_load}.json'

with open(json_file_path, 'r', encoding='utf-8') as f:
    outputs = json.load(f)

print(json.dumps(outputs, indent=4))

{
    "model_name": "microsoft/Phi-3-medium-128k-instruct",
    "generation_date": "20250723_231551",
    "results": [
        {
            "prompt": "Someone takes credit for your hard work at the office.",
            "generated_text": "\n2. You're unable to find a job after graduating college.\n3. A friend borrows your car and returns it with a scratch.\n4. You're falsely accused of a crime you didn't commit.\n\nA) Coping with financial insecurity\nB) Dealing with betrayal and loss of trust\nC) Handling damage to personal property\nD) Addressing wrongful accusations\n\nAnswers: 1-B, 2-A, 3-C, 4-D\n\nExercise 4: Describe a situation where you faced a challenge and turned it into an opportunity.\n\nAnswer: One time, I was supposed to give a presentation in front of the whole class, but I was feeling really nervous. Instead of letting my nerves get the best of me, I decided to practice my presentation multiple times and ask my teacher for feedback. When the day of the presentation arr

In [12]:
prompt_to_see = 0

print(outputs["results"][prompt_to_see]["prompt"])
print(outputs["results"][prompt_to_see]["emotion"])
print(activations[f'prompt_{prompt_to_see}_layer_10'])

Someone takes credit for your hard work at the office.
anger
[-0.06005859 -0.04858398  0.12402344 ... -0.3828125  -0.01037598
 -0.21386719]


Yo quiero terminar con un dataframe con n filas (n=n_prompts*n_layers_observadas), y tenga las columnas "prompt", "generated_text", "layer", "activation", "emotion".

In [13]:
df_base = pd.DataFrame(outputs["results"])

parsed_activations = []

    # Dynamically parse keys like 'prompt_0_layer_10'
    # The regex looks for "prompt_" followed by digits, and "layer_" followed by digits
pattern = re.compile(r'prompt_(\d+)_layer_(\d+)')

for key in activations.files:
    match = pattern.search(key)
    if match:
        prompt_index = int(match.group(1))
        layer_number = int(match.group(2))
            
        parsed_activations.append({
                "prompt_index": prompt_index,
                "layer": layer_number,
                "activation": activations[key]
            })

    # Convert the list of dicts into a "long" DataFrame
df_activations_long = pd.DataFrame(parsed_activations)
    
    # --- Step 3: Pivot the activation DataFrame from long to wide format ---
print("Pivoting activations from long to wide format...")
df_activations_wide = df_activations_long.pivot(
        index='prompt_index',   # Each row will be a unique prompt
        columns='layer',        # Each unique layer will become a column
        values='activation'     # The cell values will be the activation arrays
    )
    
    # Rename the columns to the desired format 'activation_layer_X'
df_activations_wide.columns = [f"activation_layer_{col}" for col in df_activations_wide.columns]
print(f"Created wide activation DataFrame with columns: {df_activations_wide.columns.tolist()}")

    # --- Step 4: Merge the base DataFrame with the wide activation DataFrame ---
print("Merging base data with activation data...")
    # We merge on the index. The index of df_base (0, 1, 2...) corresponds
    # to the 'prompt_index' we created for df_activations_wide.
final_df = pd.merge(
        df_base,
        df_activations_wide,
        left_index=True,    # Use the index of the left DataFrame (df_base)
        right_index=True,   # Use the index of the right DataFrame (df_activations_wide)
        how='left'          # Keep all prompts, even if some have no activations
    )

Pivoting activations from long to wide format...
Created wide activation DataFrame with columns: ['activation_layer_0', 'activation_layer_2', 'activation_layer_4', 'activation_layer_6', 'activation_layer_8', 'activation_layer_10', 'activation_layer_12', 'activation_layer_14', 'activation_layer_16', 'activation_layer_18', 'activation_layer_20', 'activation_layer_22', 'activation_layer_24', 'activation_layer_26', 'activation_layer_28', 'activation_layer_30', 'activation_layer_32', 'activation_layer_34', 'activation_layer_36', 'activation_layer_38']
Merging base data with activation data...


In [14]:
final_df.head()

,prompt,generated_text,emotion,activation_layer_0,activation_layer_2,activation_layer_4,activation_layer_6,activation_layer_8,activation_layer_10,activation_layer_12,...,activation_layer_20,activation_layer_22,activation_layer_24,activation_layer_26,activation_layer_28,activation_layer_30,activation_layer_32,activation_layer_34,activation_layer_36,activation_layer_38
0,Someone takes credit for your hard work at the...,\n2. You're unable to find a job after graduat...,anger,"[-0.00079345703, -0.0057373047, 0.0010070801, ...","[-0.007293701, -0.0071105957, 0.0040283203, 0....","[0.003967285, 0.0035247803, -0.029052734, 0.02...","[-0.016113281, 0.0005874634, -0.06591797, 0.00...","[-0.018066406, -0.051513672, -0.15722656, 0.03...","[-0.060058594, -0.048583984, 0.12402344, 0.453...","[0.12597656, 0.087402344, 0.0016174316, -0.291...",...,"[-0.016113281, 0.29882812, 0.3203125, -0.14648...","[0.14941406, -0.22949219, -0.23339844, 0.09619...","[0.57421875, 0.25390625, 0.29296875, 0.9570312...","[0.32617188, -0.5078125, 1.40625, 0.890625, -2...","[1.1875, -0.66796875, 0.625, -0.19824219, 0.96...","[0.40820312, -0.84375, -0.4609375, 0.6015625, ...","[-0.16601562, 1.65625, -1.03125, -0.33398438, ...","[-0.37890625, 0.9765625, 2.296875, -1.5, 0.417...","[-0.8203125, 0.83203125, 0.49609375, -1.21875,...","[0.34960938, 0.9453125, 0.328125, -0.59375, -1..."
1,You're cut off in traffic while driving home.,You're watching a movie and a character does ...,anger,"[-0.00079345703, -0.0057373047, 0.0010070801, ...","[-0.044189453, -0.005218506, 0.016235352, 0.02...","[0.030273438, 0.015625, -0.075683594, 0.015075...","[-0.011047363, -0.051513672, -0.01965332, 0.00...","[-0.064453125, -0.014160156, -0.027709961, -0....","[-0.010437012, -0.014892578, -0.087402344, 0.2...","[-0.09765625, -0.12890625, -0.13769531, 0.0471...",...,"[0.58984375, -0.17773438, 0.022949219, -0.8476...","[0.75390625, -0.37109375, -0.359375, 0.1914062...","[0.119140625, -0.5390625, 0.18652344, -0.02392...","[0.95703125, -0.9140625, 1.28125, 0.8359375, -...","[0.14746094, 0.22753906, -0.70703125, 1.5, 0.6...","[0.60546875, -0.41210938, -0.20019531, -0.3398...","[-0.421875, 0.1484375, -0.9765625, 0.68359375,...","[0.110839844, -0.33789062, 1.4921875, -0.02661...","[0.24902344, -0.17480469, -0.90234375, -0.4433...","[-0.41601562, -0.021240234, 0.38671875, 0.2968..."
2,You discover a scratch on your new car in the ...,What should you do? \nAnswer: Report the dama...,anger,"[-0.00079345703, -0.0057373047, 0.0010070801, ...","[-0.0067443848, -0.003829956, 0.005004883, 0.0...","[0.003540039, -0.010559082, -0.022583008, 0.02...","[-0.011962891, -0.035888672, -0.015563965, 0.0...","[-0.012451172, 0.12207031, -0.08642578, 0.0703...","[0.08300781, -0.008850098, -0.07421875, 0.1826...","[-0.009521484, 0.13769531, -0.087890625, 0.033...",...,"[0.049316406, 0.29882812, 0.6015625, 0.0854492...","[0.12988281, -0.609375, 0.44335938, -0.6679687...","[-0.40429688, -0.125, 0.17382812, -0.041748047...","[1.2734375, -0.56640625, 1.9140625, 0.13378906...","[0.77734375, -0.14550781, -0.9296875, 0.609375...","[-0.57421875, 0.41015625, -0.27148438, -0.3769...","[0.9921875, -0.41992188, -0.453125, -0.3515625...","[-0.640625, 1.1640625, 0.45507812, -1.0, -1.42...","[0.640625, -0.15820312, 0.15332031, -0.8046875...","[-0.34179688, 0.64453125, 0.06689453, -0.32031..."
3,Your reservation at a restaurant is given away...,\n\n\n### answer ###\n\nIn these two scenarios...,anger,"[-0.00079345703, -0.0057373047, 0.0010070801, ...","[0.0044555664, 0.0009460449, 0.003479004, 0.01...","[0.0029754639, -0.009338379, -0.0053100586, 0....","[0.004180908, 0.0017547607, -0.03515625, 0.003...","[0.010192871, 0.030395508, -0.0013961792, 0.09...","[0.08984375, -0.06591797, 0.13964844, 0.375, 0...","[0.19726562, 0.1171875, 0.12695312, -0.0219726...",...,"[0.22167969, -0.18554688, 0.95703125, 0.112792...","[-0.23925781, -0.16015625, -0.2734375, -0.5664...","[-0.44726562, -0.107421875, -0.23632812, 0.515...","[0.42773438, -0